# Day 15 — Simplified: Self-Attention (The ONE Idea Behind LLMs)

## The Most Important Day

If you only remember ONE thing from this whole 45-day journey, remember **self-attention**. This idea built GPT, Claude, Llama, ChatGPT. Without it, none of them exist.

## The Question

When predicting the next word in a sentence, **which earlier words matter most?**

```
"The cat that I saw yesterday sat on the mat."
                                ↑
              When the model processes "sat", which words help?
              - "cat"        → YES! That's the subject of "sat"
              - "I saw"      → less important, that's about the cat
              - "yesterday"  → barely relevant (just a time)
              - "the mat"    → important (it's WHERE)
```

The model needs to **pick and choose** which words to focus on. That's attention.

## The Buffet Analogy

Imagine each word is hungry and standing at a buffet. The other words are dishes on the buffet.

```
Each word asks itself: "Of all these dishes, which match what I want?"

It LOOKS at every dish, RATES each one ("interest level"),
then takes proportional bites from each.
```

This is exactly self-attention. Each word looks at every other word, decides how much to "take" from each, and combines them.

## The Three Things Each Word Computes

Every word makes 3 vectors out of itself:

```
Query (Q)  — "What am I looking for?"      (e.g., "I need a subject")
Key (K)    — "What can I offer?"           (e.g., "I'm a noun, gender singular")
Value (V)  — "What info do I carry?"       (the actual meaningful content)
```

These are produced by 3 separate linear layers from the word's embedding.

## The 4 Steps of Attention

```
1. SCORE
   For each word, compute its Q · K with every other word.
   Big dot product = strong match = high attention.

2. SCALE
   Divide scores by √d (square root of head dim).
   Why? Keeps numbers tame as dimensions grow.

3. SOFTMAX
   Turn scores into weights that sum to 1 (per row).
   "How much of my attention goes to each word?"

4. COMBINE
   Take weighted average of every word's Value.
   This is what each word "becomes" after attention.
```

## The "Causal Mask" — No Peeking at the Future

For LLMs that PREDICT the next token, the model can only look BACK, never forward. We enforce this by setting "future" attention scores to negative infinity before softmax. After softmax, they become zero.

```
Position 0 can see: {0}
Position 1 can see: {0, 1}
Position 2 can see: {0, 1, 2}
Position 3 can see: {0, 1, 2, 3}
```

Visually: a triangular mask.

## Why This is Magic

Compared to RNNs:

```
RNN:                          Self-attention:

Process tokens in order       Process ALL tokens at once
(SLOW, sequential)            (FAST, parallel — perfect for GPUs)

Forgets long-range info       Looks at any past position directly
(vanishing gradients)         (no forgetting!)

Hidden state is opaque        Attention weights are VISIBLE
                              (we can see what the model is looking at)
```

Attention won. Every modern LLM uses it.

## The Whole Formula (Memorize This)

```
Attention(Q, K, V) = softmax(Q @ K.T / √d) @ V
```

That's it. The most important formula in modern AI fits in one line.

## What's Next

Today: ONE attention head.
Day 16: MULTIPLE attention heads in parallel (different "specialists").
Day 17: Add position info (because attention itself has no sense of order!).
Day 18: Build a full text generator using all of this.
Day 19+: Wrap it into a "transformer block," stack many → mini GPT.

See `notebook.ipynb` for a hand-worked example with tiny numbers + full code.

In [ ]:
import torch
import torch.nn.functional as F

# A FULL attention computation in 5 lines
# (using tiny numbers so you can see everything)

# 3 tokens, each 2-dimensional
X = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

# Step 0: Use X itself for Q, K, V (real models project these separately)
Q, K, V = X, X, X

# Step 1: Score
scores = Q @ K.T            # shape (3, 3)
print(f"Step 1 — Scores:\n{scores}\n")

# Step 2: Scale
scaled = scores / (Q.shape[-1] ** 0.5)
print(f"Step 2 — Scaled (divided by √d):\n{scaled}\n")

# Step 3: Softmax (each row sums to 1)
weights = F.softmax(scaled, dim=-1)
print(f"Step 3 — Softmax weights (each row sums to 1):\n{weights}\n")

# Step 4: Combine — weighted sum of values
output = weights @ V
print(f"Step 4 — Output (weighted average of V):\n{output}")

## Read the output:

Each row of `weights` is one token saying "Here's how much attention I pay to each other token."

For instance: row 0 (token 0) might say [0.42, 0.16, 0.42] = "I pay 42% attention to myself, 16% to token 1, 42% to token 2."

The OUTPUT row 0 is then a weighted average of all 3 input vectors using those weights.

**Every word ends up as a blend of all the words it cared about.**

That's self-attention. Stack 12 of these layers + add the right plumbing = you have GPT.